# 大模型推理优化：从自回归执行到集群性能

> **本章定位**：承接 [60_inference_deployment.ipynb](60_inference_deployment.ipynb) 建立的生产推理主链路，在模型产物与服务契约确定后，优化 Prefill、Decode、KV Cache、Attention Kernel、编译、调度和推理并行。

> **章节边界**：本章属于训练与推理系统：推理优化；制品发布、API、扩缩容、可观测性与故障恢复由 [60_inference_deployment.ipynb](60_inference_deployment.ipynb) 承接，量化、剪枝、蒸馏和低秩分解由 [A60_model_compression.ipynb](A60_model_compression.ipynb) 承接。本章聚焦模型产物在目标推理 Kernel 中的执行方式。

**本章总览**：使用支持 KV Cache 的最小 Causal LM 建立可解释基线，依次分析 Prefill/Decode、缓存、批处理、Attention Kernel、编译、推测解码、性能基准与分布式执行模式。


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 训练与推理系统：推理优化 |
| 本章定位 | 纵向扩展推理性能，从朴素自回归执行进入缓存、Kernel、调度和并行。 |
| 先修知识 | 掌握 `31` 的自回归生成和 `60` 的服务契约。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | 核心实验 CPU/Apple Silicon 可运行；Kernel 性能需目标 GPU。 |
| 输入 | 固定模型、Tokenizer、生成配置和真实长度分布。 |
| 交付物 | KV Cache 实现、性能基线、优化矩阵和部署决策顺序。 |

### 1.1．学习目标

完成本章后，读者能够解释 Prefill 与 Decode 的成本差异，验证 KV Cache 和批调度语义，比较 Attention、编译、量化与推测解码路径，并以 TTFT、TPOT、吞吐、显存和单位有效 Token 成本选择推理拓扑。


In [ ]:
# 基础环境。后续主路径只依赖这些包。
import copy
import math
import platform
import random
import statistics
import time
from dataclasses import dataclass
from typing import Iterable, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# 42 仅固定随机模型与输入；优化对照使用同一组预注册 Seed，跨后端或 Kernel 不保证逐 bit 一致。
SEED = 42
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": str(DEVICE),
})


### 1.2．环境与依赖

- Transformers：`generate()`、缓存策略与 Attention backend。
- bitsandbytes：受支持 CUDA 环境中的 INT8/INT4 权重量化。
- flash-attn：仅在模型、GPU、CUDA 与 PyTorch 版本共同兼容时使用。


In [ ]:
# 按需取消注释；安装后重启 Kernel。
# %pip install -U transformers accelerate
# %pip install -U bitsandbytes
# %pip install -U flash-attn --no-build-isolation


### 1.3．实验基座

本章提供 Tokenizer、数据整理和支持 KV Cache 的最小 Causal LM，不依赖训练优化 Notebook。随机初始化模型不用于评估语言能力，其作用是验证 Token 一致性、内存公式、延迟与吞吐测量方法。


In [ ]:
# 小型指令数据；生产数据应做去重、质量过滤、隐私与许可证审查。
records = [
    {"prompt": "把 2 加 3 等于多少？", "response": "2 加 3 等于 5。"},
    {"prompt": "把 7 加 4 等于多少？", "response": "7 加 4 等于 11。"},
    {"prompt": "什么是梯度累积？", "response": "梯度累积把多个微批次的梯度合并后再更新参数。"},
    {"prompt": "什么是混合精度？", "response": "混合精度用较低精度计算，并在关键位置保留足够精度。"},
    {"prompt": "LoRA 的核心思想是什么？", "response": "LoRA 用两个低秩矩阵学习权重增量，并冻结原始权重。"},
    {"prompt": "KV Cache 有什么作用？", "response": "KV Cache 复用历史 Token 的键和值，避免重复计算。"},
    {"prompt": "什么是吞吐量？", "response": "吞吐量表示单位时间内系统处理的 Token 或请求数量。"},
    {"prompt": "什么是首 Token 延迟？", "response": "首 Token 延迟是请求到达后生成第一个 Token 所需的时间。"},
]

# 前 6 条训练、后 2 条验证沿用训练章的数据契约；数据或模板变化后优化前后结果不可直接比较。
train_records = records[:6]
valid_records = records[6:]


In [ ]:
# 最小字符级 Tokenizer。
class MyCharTokenizer:
    """基于给定文本构建带特殊符号的最小字符级词表，并提供字符与 Token ID 的双向转换。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, texts: Iterable[str]):
        """汇总语料字符并建立稳定的正反向词表，同时记录 PAD、BOS、EOS 与 UNK 的 ID。"""
        specials = ["<pad>", "<bos>", "<eos>", "<unk>"]
        chars = sorted(set("".join(texts)))
        self.id_to_token = specials + chars
        self.token_to_id = {token: idx for idx, token in enumerate(self.id_to_token)}
        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    @property
    def vocab_size(self) -> int:
        """返回包含特殊符号在内的词表大小。"""
        return len(self.id_to_token)

    def encode(self, text: str) -> list[int]:
        """将字符串逐字符映射为 Token ID，未登录字符回退到 UNK。"""
        return [self.token_to_id.get(char, self.unk_token_id) for char in text]

    def decode(self, ids: Iterable[int], skip_special_tokens: bool = True) -> str:
        """将 Token ID 序列还原为字符串，并可过滤特殊符号。"""
        specials = {"<pad>", "<bos>", "<eos>", "<unk>"}
        tokens = [self.id_to_token[int(idx)] for idx in ids]
        if skip_special_tokens:
            tokens = [token for token in tokens if token not in specials]
        return "".join(tokens)


def my_format_prompt(prompt: str) -> str:
    """把用户问题格式化为训练与推理共用的固定对话模板。"""
    return f"用户：{prompt}\n助手："


all_texts = [
    text
    for record in records
    for text in (my_format_prompt(record["prompt"]), record["response"])
]
tokenizer = MyCharTokenizer(all_texts)

probe = "什么是吞吐量？"
print("vocab_size =", tokenizer.vocab_size)
print(tokenizer.encode(probe), "->", tokenizer.decode(tokenizer.encode(probe)))


In [ ]:
# 单样本编码与动态 Padding Collator。
# -100 对齐 CrossEntropy ignore_index，128 Token 上限绑定小模型上下文；模板或截断变化须同步回归基线。
IGNORE_INDEX = -100


def my_encode_sft_record(record: dict, max_length: int = 128) -> dict[str, list[int]]:
    """把一条问答记录编码为因果语言模型序列，并用 -100 屏蔽 BOS 与提示词标签；结果会截断到 max_length。"""
    prompt_ids = tokenizer.encode(my_format_prompt(record["prompt"]))
    answer_ids = tokenizer.encode(record["response"])
    input_ids = [tokenizer.bos_token_id] + prompt_ids + answer_ids + [tokenizer.eos_token_id]
    labels = [IGNORE_INDEX] * (1 + len(prompt_ids)) + answer_ids + [tokenizer.eos_token_id]

    input_ids = input_ids[:max_length]
    labels = labels[:max_length]
    return {"input_ids": input_ids, "labels": labels}


class MySFTCollator:
    """对长度不同的 SFT 样本执行动态右填充，生成 input_ids、attention_mask 与 labels 批张量。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, pad_token_id: int, pad_to_multiple_of: Optional[int] = None):
        """保存填充 Token ID 及可选的长度对齐倍数。"""
        self.pad_token_id = pad_token_id
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, examples: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
        """将样本填充到批内最大长度或指定倍数，标签填充值使用 IGNORE_INDEX。"""
        max_len = max(len(example["input_ids"]) for example in examples)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            max_len = math.ceil(max_len / m) * m

        input_ids, attention_mask, labels = [], [], []
        # 逐条处理样本，并把结果汇总到统一的数据结构中。
        for example in examples:
            pad_len = max_len - len(example["input_ids"])
            input_ids.append(example["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(example["input_ids"]) + [0] * pad_len)
            labels.append(example["labels"] + [IGNORE_INDEX] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def my_move_batch(batch: dict[str, torch.Tensor], device: torch.device):
    """把批字典中的所有张量移动到指定计算设备，并返回新的字典。"""
    return {name: tensor.to(device) for name, tensor in batch.items()}


encoded_train = [my_encode_sft_record(record) for record in train_records]
encoded_valid = [my_encode_sft_record(record) for record in valid_records]
collator = MySFTCollator(tokenizer.pad_token_id)
batch = collator(encoded_train[:3])

print({name: tuple(value.shape) for name, value in batch.items()})


In [ ]:
# 可同时用于训练与后续 KV Cache 验证的最小注意力层。
class MyCausalSelfAttention(nn.Module):
    """实现支持 Padding Mask 与 KV Cache 的多头因果自注意力。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        """创建融合 QKV 投影、输出投影，并记录注意力头布局与 Dropout。"""
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(
        self,
        x: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> tuple[torch.Tensor, Optional[tuple[torch.Tensor, torch.Tensor]]]:
        """计算因果掩码注意力，必要时拼接历史 Key/Value；返回 [B,T,D] 输出及可选的新缓存。"""
        batch_size, query_len, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, query_len, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (tensor.transpose(1, 2) for tensor in (q, k, v))  # [B, H, T, D]

        past_len = 0
        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(2)
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        key_len = k.size(2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)

        query_positions = past_len + torch.arange(query_len, device=x.device)
        key_positions = torch.arange(key_len, device=x.device)
        causal_mask = key_positions[None, :] > query_positions[:, None]
        scores = scores.masked_fill(causal_mask[None, None, :, :], torch.finfo(scores.dtype).min)

        if attention_mask is not None:
            if attention_mask.size(1) != key_len:
                raise ValueError(f"attention_mask 长度应为 {key_len}，实际为 {attention_mask.size(1)}")
            key_padding_mask = attention_mask[:, None, None, :] == 0
            scores = scores.masked_fill(key_padding_mask, torch.finfo(scores.dtype).min)

        probs = F.softmax(scores, dim=-1)
        probs = F.dropout(probs, p=self.dropout, training=self.training)
        out = probs @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, query_len, d_model)
        present = (k, v) if use_cache else None
        return self.proj(out), present


class MyTransformerBlock(nn.Module):
    """组合 Pre-Norm 因果自注意力、前馈网络与两条残差连接。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: int, dropout: float):
        """按模型宽度、头数和 MLP 扩展比创建 Transformer Block 子层。"""
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MyCausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Linear(mlp_ratio * d_model, d_model),
            nn.Dropout(dropout),
        )

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(self, x, attention_mask=None, past_key_value=None, use_cache=False):
        """依次执行注意力与 MLP 残差更新，并透传可选 KV Cache。"""
        attn_out, present = self.attn(
            self.ln1(x),
            attention_mask=attention_mask,
            past_key_value=past_key_value,
            use_cache=use_cache,
        )
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, present


In [ ]:
# 最小 Causal LM；内置梯度检查点开关，后续再启用。
from torch.utils.checkpoint import checkpoint


@dataclass
class MyCausalLMOutput:
    """承载因果语言模型的 logits、可选损失与逐层 KV Cache。"""
    # 执行前向计算，得到后续损失或解码需要的模型输出。
    logits: torch.Tensor
    loss: Optional[torch.Tensor] = None
    past_key_values: Optional[list[tuple[torch.Tensor, torch.Tensor]]] = None


class MyTinyCausalLM(nn.Module):
    """实现带位置嵌入、权重共享、可选 KV Cache 和标签损失的最小 Decoder-only 语言模型。"""
    # 在初始化阶段注册参数、子层和不会随批次变化的配置。
    # d_model=64、4 头、2 层、4×MLP 与 256 位置构成带 KV Cache 的小模型；宽度须整除头数。
    # 该规模只验证机制，瓶颈比例不能外推到真实大模型。
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        max_length: int = 256,
        dropout: float = 0.0,
    ):
        """创建 Token/位置嵌入、Transformer Block、最终归一化和共享权重输出头。"""
        super().__init__()
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.blocks = nn.ModuleList([
            MyTransformerBlock(d_model, n_heads, mlp_ratio=4, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight  # Weight tying（权重共享）
        self.gradient_checkpointing = False

    # 前向传播按照本模块的数据流连接各子层，并返回当前阶段输出。
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        past_key_values: Optional[list[tuple[torch.Tensor, torch.Tensor]]] = None,
        use_cache: bool = False,
    ) -> MyCausalLMOutput:
        """把 [B,T] Token 序列映射为 [B,T,V] logits，并可计算移位交叉熵或返回逐层缓存；超长序列会报错。"""
        batch_size, query_len = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + query_len > self.max_length:
            raise ValueError("序列超过 max_length")

        positions = torch.arange(past_len, past_len + query_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        presents = [] if use_cache else None

        for layer_idx, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[layer_idx]
            if self.gradient_checkpointing and self.training:
                if use_cache:
                    raise ValueError("训练时梯度检查点与 KV Cache 不应同时启用")
                x, _ = checkpoint(
                    lambda hidden, current_block=block: current_block(
                        hidden, attention_mask=attention_mask
                    )[0],
                    x,
                    use_reentrant=False,
                ), None
            else:
                x, present = block(
                    x,
                    attention_mask=attention_mask,
                    past_key_value=past,
                    use_cache=use_cache,
                )
                if use_cache:
                    presents.append(present)

        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=IGNORE_INDEX,
            )
        return MyCausalLMOutput(logits=logits, loss=loss, past_key_values=presents)


model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable parameters: {trainable:,}")


In [ ]:
# 部署实验使用随机初始化小模型，只验证机制与相对性能。
baseline_model = MyTinyCausalLM(tokenizer.vocab_size).to(DEVICE).eval()
probe_batch = {
    name: tensor[:2].to(DEVICE)
    for name, tensor in batch.items()
}
print({
    "parameters": sum(parameter.numel() for parameter in baseline_model.parameters()),
    "device": str(DEVICE),
    "semantic_quality": "随机模型无语言能力；仅用于正确性与性能机制验证",
})


In [ ]:
# 为 Transformers generate() 创建本地随机小模型配置，不下载权重。
try:
    from transformers import LlamaConfig, LlamaForCausalLM

    # 集中定义结构和运行参数，避免配置散落在后续逻辑中。
    hf_config = LlamaConfig(
        vocab_size=tokenizer.vocab_size,
        hidden_size=64,
        intermediate_size=128,
        num_hidden_layers=2,
        num_attention_heads=4,
        num_key_value_heads=2,
        max_position_embeddings=256,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    print("本地 Transformers 配置已创建。")
except ImportError as exc:
    print("跳过：需要 transformers。", exc)


## 2．直觉与输入输出契约

推理链路需要区分 Prefill 与 Decode，并沿“减少重复工作 → 加快 Kernel → 控制形状与编译 → 提高批处理利用率 → 扩展推理并行”逐层优化。

<!-- diagram:inference-optimization-route -->

![架构图：在线推理中 Scheduler、Prefill、KV Block、Decode 与流式输出的数据面](assets/figures/A50_inference_optimization/inference-optimization-route.svg)

[TikZ 源文件](assets/figures/A50_inference_optimization/inference-optimization-route.tex)

部署控制面负责版本、鉴权、发布、扩缩容与故障恢复；本章聚焦这条数据面的性能。


### 2.1．推理优化方法图谱

推理优化应围绕 SLO 和真实负载分类。模型压缩产物是本章输入之一；剪枝、蒸馏、低秩和量化算法由模型压缩章集中讨论，本章不重复实现。量化只有命中目标硬件 Kernel 时才形成推理加速。

```mermaid
flowchart TD
    S["推理目标<br/>质量、TTFT、TPOT、吞吐、成本"] --> B{"主要瓶颈"}
    B -->|"重复计算"| R["以存代算<br/>KV Cache、Prefix Cache、Response Cache"]
    B -->|"串行 Decode 步数"| D["Speculative Decoding<br/>Draft + Verify"]
    B -->|"Kernel / HBM / Python 开销"| K["低精度、SDPA / FlashAttention<br/>融合算子、编译、CUDA Graph"]
    B -->|"GPU 利用率不足"| Q["Continuous Batching、Paged KV<br/>Chunked Prefill、长度分桶"]
    B -->|"模型或 KV 放不下"| M["量化推理格式、GQA / MQA<br/>KV 分层、TP / PP、Offload"]
    B -->|"集群与服务瓶颈"| C["Replica、P/D 分离<br/>Cache 分层、路由、背压"]
```

| 优化层次 | 代表方法 | 主要减少什么 | 主要代价 |
|---|---|---|---|
| 计算复用 / 以存代算 | KV Cache、Prefix Cache、Response Cache | 历史 token、公共前缀或完整响应的重复计算 | 显存/存储、失效策略、隐私与一致性 |
| 解码算法 | Speculative Decoding、并行候选验证 | 大模型串行 Forward 次数 | Draft 成本、接受率、调度复杂度 |
| 数值与 Kernel | BF16/FP16/FP8/INT8/INT4、SDPA、FlashAttention、融合算子 | 算术成本、HBM 访问和 Kernel Launch | 硬件与形状约束、精度回归 |
| 编译与形状 | torch.compile、CUDA Graph、Static Cache、形状分桶 | Python/调度开销和重复编译 | 冷启动、内存预分配、动态图回退 |
| 调度与显存管理 | Dynamic/Continuous Batching、Paged KV、Chunked Prefill | Padding、显存碎片和 GPU 空洞 | 公平性、抢占、尾延迟 |
| 上游模型压缩产物 | 量化权重、更小 Student、结构化稀疏模型 | 参数、带宽或实际 FLOPs | 必须匹配模型格式与推理 Kernel |
| 高效模型结构 | GQA/MQA、Sliding Window、Sparse Attention、MoE | KV 或每 token 激活计算 | 架构重训、覆盖范围或通信复杂度 |
| 分布式服务 | Replica、TP、PP、EP、P/D 分离、Offload | 单卡容量或单实例吞吐限制 | Collective、网络、路由与故障域 |

三个常见误区：

- **Paged KV 不等于 KV Cache**：KV Cache 决定复用什么；Paged KV 解决这些 Cache Block 如何分配、复用和回收。
- **存算分离不等于以存代算**：以存代算保留中间结果以避免重算；存算分离把权威存储、缓存和计算实例解耦。
- **MoE 不等于模型压缩**：MoE 通常增加总参数，只让每个 token 激活少量 Experts。

模型压缩的方法选择与量化格式见唯一的压缩主章 [A60_model_compression.ipynb](A60_model_compression.ipynb)。


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

KV Cache 复用历史 Token 的键和值，避免在每个 Decode 步骤重复投影，但历史 Cache 仍需被读取。忽略对齐和分页元数据时，其显存近似为：

$$
M_{\mathrm{KV}}\approx 2B L N_{\mathrm{layers}}N_{kv}d_h\frac{b}{8}
$$

其中，$B$ 是并发序列数，$L$ 是已缓存长度，$N_{kv}$ 是 KV 头数，$d_h$ 是每头维度，$b$ 是元素位数，系数 $2$ 表示 K 与 V。张量并行通常要求 $N_q\bmod TP=0$；若 KV 头按 Rank 切分，还要求 $N_{kv}\bmod TP=0$。支持 KV 头复制的 GQA/MQA Runtime 可放宽后一约束，但会改变显存和通信成本。`past_key_values`、Paged KV 分配器与服务指标分别对应数学状态、生产存储和实测校准。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．朴素自回归生成

自回归生成（Autoregressive Generation）每次根据已有序列预测下一个 Token，并将结果追加到输入。模型接收提示 Token 与最大新 Token 数，输出提示和生成 Token 组成的完整序列。该实现语义正确但存在重复计算：第 $t$ 步会重新处理此前的全部 Token。固定贪心解码后，后续 KV Cache 版本应生成完全一致的 Token 序列。


In [ ]:
# 朴素 Greedy Decoding（贪心解码）。
@torch.inference_mode()
def my_generate_naive(
    model: MyTinyCausalLM,
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    eos_token_id: Optional[int] = None,
) -> torch.Tensor:
    """以贪心策略逐 Token 生成，每一步都重算完整前缀；返回包含原提示的 Token 序列。"""
    model.eval()
    generated = prompt_ids
    for _ in range(max_new_tokens):
        attention_mask = torch.ones_like(generated)
        # 执行前向计算，得到后续损失或解码需要的模型输出。
        logits = model(generated, attention_mask=attention_mask).logits
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break
    return generated


prompt = my_format_prompt("什么是梯度累积？")
prompt_tensor = torch.tensor(
    [[tokenizer.bos_token_id] + tokenizer.encode(prompt)],
    dtype=torch.long,
    device=DEVICE,
)
# 8 个新 Token 用于缓存正确性观察；4/8/16 用于短基准，64 用于长度上限，生成越长 Decode 与 KV 近似线性增长。
naive_ids = my_generate_naive(baseline_model, prompt_tensor, max_new_tokens=8, eos_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(naive_ids[0].tolist()))


### 3.2．KV Cache、Prefix Cache 与响应缓存

KV Cache 保存每层历史 token 的 Key 和 Value，使 Decode 每步只为新 token 计算 K/V。它是单请求内部的以存代算：

- **Prefill**：并行处理完整 Prompt，建立 KV Cache，主要影响 TTFT。
- **Decode**：复用历史 K/V，只追加新 token，主要影响 TPOT。
- 不保存历史 Query，因为旧位置的 Query 不会再次用于生成新位置。

以存代算可以向更高层扩展：

| 复用层级 | 缓存内容 | 命中时省掉什么 | 主要风险 |
|---|---|---|---|
| 单请求 KV Cache | 当前请求各层 K/V | 历史 token 的 K/V 投影和 Attention 历史重算 | 随上下文、层数、并发线性增长 |
| 跨请求 Prefix Cache | 公共 Prompt 前缀对应的 KV Blocks | 重复 System Prompt、长文档前缀的 Prefill | Cache Key 必须包含模型、Adapter、Tokenizer 和前缀 token |
| Response Cache | 完整或模板化请求的最终响应 | 整次模型调用 | 随机采样、时效性、权限、隐私和工具状态 |
| Semantic Cache | 语义近似请求与响应 | 相似请求的整次调用 | 误命中会改变语义，只适合可容忍场景 |

```mermaid
flowchart LR
    Q["新请求"] --> R{"Response Cache 命中？"}
    R -->|"是"| O["直接返回受控缓存响应"]
    R -->|"否"| P{"Prefix Cache 命中？"}
    P -->|"是"| K["复用公共前缀 KV Blocks"]
    P -->|"否"| F["Prefill 构建 KV Cache"]
    F --> K
    K --> D["逐 token Decode<br/>持续追加 KV"]
    D --> O
```

Paged KV / PagedAttention 解决 KV Block 的非连续分配、共享和回收，主要减少显存碎片并支持 Prefix Cache；它不是另一种 Attention 数学。Static Cache 预分配最大空间，便于编译和 CUDA Graph，但会牺牲动态内存效率。KV Quantization、CPU/NVMe Offload 和远端 KV Store 则是在容量、带宽与延迟之间继续折中。


In [ ]:
# 使用 MyTinyCausalLM 已实现的 past_key_values。
@torch.inference_mode()
def my_generate_cached(
    model: MyTinyCausalLM,
    prompt_ids: torch.Tensor,
    max_new_tokens: int,
    eos_token_id: Optional[int] = None,
) -> torch.Tensor:
    """先执行 Prefill，再复用逐层 KV Cache 贪心解码；返回包含原提示的 Token 序列。"""
    model.eval()
    generated = prompt_ids
    attention_mask = torch.ones_like(generated)
    # 执行前向计算，得到后续损失或解码需要的模型输出。
    output = model(generated, attention_mask=attention_mask, use_cache=True)  # Prefill
    past = output.past_key_values

    # 逐步执行训练或生成流程，并在每一步更新当前状态。
    for step in range(max_new_tokens):
        next_token = output.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
        if eos_token_id is not None and torch.all(next_token == eos_token_id):
            break
        if step + 1 < max_new_tokens:
            attention_mask = torch.ones_like(generated)
            output = model(
                next_token,
                attention_mask=attention_mask,
                past_key_values=past,
                use_cache=True,
            )
            past = output.past_key_values
    return generated


cached_ids = my_generate_cached(baseline_model, prompt_tensor, max_new_tokens=8, eos_token_id=tokenizer.eos_token_id)
print("naive token IDs：", naive_ids.tolist())
print("cached token IDs：", cached_ids.tolist())


#### 3.2.1．历史不重投影，但仍被读取

学习问题是：KV Cache 究竟减少了哪一部分计算。下面不另写生成算法，而是在上一单元的朴素生成与缓存生成期间记录第一层 QKV 投影的真实输入长度，并从该层实际返回的 `present` 读取 Cache 长度。四幅连续分镜保持同一批 Token 的位置不变：上轨中每一步都重新投影完整前缀；下轨在 Prefill 后只投影一个新 Token，但这个 Query 仍读取全部历史 Key/Value。

验收条件包括：两条路径生成完全相同的 Token ID；朴素路径的投影长度随前缀增长；缓存路径的 Decode 投影长度恒为 1；Cache 长度每步增加 1。图中只保留每一步末尾最多 12 个位置，省略的更早位置以省略号表示，数值摘要仍报告完整长度。


In [ ]:
# 通过 Hook 记录现有生成函数的第一层 QKV 投影长度与实际 Cache 长度。
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def my_capture_kv_trace(generate_fn, model, prompt_ids, max_new_tokens):
    """执行既有生成函数并记录第一层真实 QKV 输入长度与返回的 KV Cache 长度。"""
    query_lengths = []
    cache_lengths = []

    def record_query_length(module, inputs):
        query_lengths.append(int(inputs[0].shape[1]))

    def record_cache_length(module, inputs, output):
        present = output[1]
        cache_lengths.append(None if present is None else int(present[0].shape[2]))

    attention = model.blocks[0].attn
    query_handle = attention.qkv.register_forward_pre_hook(record_query_length)
    cache_handle = attention.register_forward_hook(record_cache_length)
    try:
        token_ids = generate_fn(
            model, prompt_ids, max_new_tokens=max_new_tokens, eos_token_id=None
        ).detach().cpu()
    finally:
        query_handle.remove()
        cache_handle.remove()

    return {
        "token_ids": token_ids,
        "query_lengths": query_lengths,
        "cache_lengths": cache_lengths,
    }


TRACE_NEW_TOKENS = 4  # 四个 Decode 时钟足以显示投影长度分叉，同时限制图形宽度。
naive_trace = my_capture_kv_trace(
    my_generate_naive, baseline_model, prompt_tensor, TRACE_NEW_TOKENS
)
cached_trace = my_capture_kv_trace(
    my_generate_cached, baseline_model, prompt_tensor, TRACE_NEW_TOKENS
)
prompt_length = int(prompt_tensor.shape[1])
expected_prefix_lengths = [prompt_length + step for step in range(TRACE_NEW_TOKENS)]
expected_cached_queries = [prompt_length] + [1] * (TRACE_NEW_TOKENS - 1)

if not torch.equal(naive_trace["token_ids"], cached_trace["token_ids"]):
    raise RuntimeError("朴素路径与 KV Cache 路径生成了不同的 Token ID")
if naive_trace["query_lengths"] != expected_prefix_lengths:
    raise RuntimeError(f"朴素投影长度不符合完整前缀契约：{naive_trace['query_lengths']}")
if cached_trace["query_lengths"] != expected_cached_queries:
    raise RuntimeError(f"缓存 Decode 的 Query 长度不符合契约：{cached_trace['query_lengths']}")
if cached_trace["cache_lengths"] != expected_prefix_lengths:
    raise RuntimeError(f"KV Cache 长度未逐步增长：{cached_trace['cache_lengths']}")

VIZ = {
    "background": "#0B1020",
    "foreground": "#E6EDF7",
    "muted": "#8792A8",
    "project": "#F2A65A",
    "cache": "#7E6FD8",
    "read": "#55C1A7",
}
DISPLAY_POSITIONS = 12  # 仅裁剪画面，不改变完整 Trace 与验收数值。
fig, axes = plt.subplots(
    2, TRACE_NEW_TOKENS, figsize=(15, 6.2), facecolor=VIZ["background"]
)

for step in range(TRACE_NEW_TOKENS):
    prefix_length = expected_prefix_lengths[step]
    first_visible = max(0, prefix_length - DISPLAY_POSITIONS)
    visible_positions = list(range(first_visible, prefix_length))

    for row, axis in enumerate(axes[:, step]):
        axis.set_facecolor(VIZ["background"])
        axis.set_xlim(-1.2, len(visible_positions) + 0.5)
        axis.set_ylim(-0.05, 1.15)
        axis.axis("off")
        if first_visible:
            axis.text(-0.8, 0.36, "...", color=VIZ["muted"], ha="center")

        for slot, position in enumerate(visible_positions):
            is_newest = position == prefix_length - 1
            if row == 0 or step == 0:
                color = VIZ["project"]
            else:
                color = VIZ["project"] if is_newest else VIZ["cache"]
            tile = FancyBboxPatch(
                (slot, 0.22), 0.78, 0.30,
                boxstyle="round,pad=0.02,rounding_size=0.05",
                facecolor=color, edgecolor=VIZ["foreground"], linewidth=0.7,
            )
            axis.add_patch(tile)
            axis.text(
                slot + 0.39, 0.37, str(position), ha="center", va="center",
                color=VIZ["background"], fontsize=8, fontweight="bold",
            )

        if row == 0:
            axis.set_title(
                f"step {step} · project {naive_trace['query_lengths'][step]} tokens",
                color=VIZ["foreground"], fontsize=10, pad=5,
            )
        else:
            query_length = cached_trace["query_lengths"][step]
            phase = "prefill" if step == 0 else "decode"
            axis.set_title(
                f"{phase} · project {query_length} · cache {prefix_length}",
                color=VIZ["foreground"], fontsize=10, pad=5,
            )
            newest_slot = len(visible_positions) - 1
            if step == 0:
                axis.text(
                    newest_slot / 2 + 0.39, 0.96, f"Q × {query_length}",
                    color=VIZ["project"], ha="center", va="center",
                    fontweight="bold",
                )
            else:
                axis.text(
                    newest_slot + 0.39, 0.96, "Q", color=VIZ["project"],
                    ha="center", va="center", fontweight="bold",
                )
                for target_slot in sorted({0, newest_slot}):
                    axis.add_patch(FancyArrowPatch(
                        (newest_slot + 0.39, 0.87), (target_slot + 0.39, 0.57),
                        arrowstyle="-|>", mutation_scale=9, linewidth=1.2,
                        color=VIZ["read"], connectionstyle="arc3,rad=0.12",
                    ))

axes[0, 0].text(
    -1.08, 0.37, "NAIVE", color=VIZ["foreground"], rotation=90,
    ha="center", va="center", fontweight="bold",
)
axes[1, 0].text(
    -1.08, 0.37, "KV CACHE", color=VIZ["foreground"], rotation=90,
    ha="center", va="center", fontweight="bold",
)
fig.suptitle(
    "KV Cache changes projection work, not attention visibility",
    color=VIZ["foreground"], fontsize=15, fontweight="bold",
)
fig.text(
    0.5, 0.02,
    "orange = projected now · purple = reused K/V · green arrows = current Q still reads history",
    ha="center", color=VIZ["muted"], fontsize=9,
)
plt.tight_layout(rect=(0.03, 0.06, 1, 0.92))
plt.show()

print({
    "same_token_ids": True,
    "naive_qkv_input_lengths": naive_trace["query_lengths"],
    "cached_qkv_input_lengths": cached_trace["query_lengths"],
    "cached_kv_lengths": cached_trace["cache_lengths"],
})


图中下轨的历史 Token 没有再次进入 QKV 投影，但当前 Query 仍与全部历史 Key 计算 Attention；因此 KV Cache 消除的是历史 K/V 的重复投影与历史状态重算，并不会把标准全注意力的单步读取成本变成常数。该分镜只验证固定小模型与贪心解码下的执行语义，不能替代目标硬件上的 TTFT、TPOT、显存和吞吐基准。


In [ ]:
# 同环境相对耗时；小模型/短序列上 Python 开销可能掩盖收益。
from tqdm.auto import tqdm, trange
def my_synchronize(device: torch.device):
    """等待 CUDA 或 MPS 上的异步算子完成，使后续计时覆盖真实设备执行。"""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


# 1 次预热、5 次重复取中位数是低成本基线；生产应在稳定点报告 P50/P95/P99 与置信区间。
def my_median_runtime_ms(fn, repeats: int = 5, warmups: int = 1) -> float:
    """完成预热并重复执行可调用对象，设备同步后返回耗时中位数，单位为毫秒。"""
    # 先预热再计时，设备同步确保测量覆盖真实算子完成时间。
    for _ in trange(
        warmups, desc="性能基准预热", unit="run", leave=False, dynamic_ncols=True
    ):
        fn()
    my_synchronize(DEVICE)
    samples = []
    for _ in trange(
        repeats, desc="性能基准采样", unit="run", leave=False, dynamic_ncols=True
    ):
        start = time.perf_counter()
        fn()
        my_synchronize(DEVICE)
        samples.append((time.perf_counter() - start) * 1000)
    return statistics.median(samples)


naive_ms = my_median_runtime_ms(lambda: my_generate_naive(baseline_model, prompt_tensor, 16))
cached_ms = my_median_runtime_ms(lambda: my_generate_cached(baseline_model, prompt_tensor, 16))
print({"naive_ms": naive_ms, "cached_ms": cached_ms, "speedup": naive_ms / cached_ms})


### 3.3．推理内存预算

对 Decoder-only Transformer，常见近似：

$$\text{Weight Bytes} \approx N_{params} \times \frac{b_w}{8}$$

$$\text{KV Bytes} = L \times 2 \times B \times S \times H_{kv} \times D_h \times \text{bytes(dtype)}$$

其中 $L$ 是层数，2 表示 K 与 V，$B$ 是并发序列数，$S$ 是已缓存长度，$H_{kv}$ 是 KV Head 数，$D_h$ 是 Head Dimension。GQA（Grouped-Query Attention，分组查询注意力）通过减少 $H_{kv}$ 降低 Cache。

容量估算还需为未量化模块、量化 scale、CUDA context、激活、logits、采样缓冲、通信缓冲、内存碎片与运行时工作区留余量。理论值不能直接作为可分配上限。


In [ ]:
# 权重与 KV Cache 估算器。
def my_gib(num_bytes: float) -> float:
    """把字节数换算为二进制 GiB。"""
    return num_bytes / 2**30


def my_estimate_inference_memory(
    parameters: int,
    weight_bits: int,
    layers: int,
    batch_size: int,
    sequence_length: int,
    kv_heads: int,
    head_dim: int,
    kv_bytes_per_element: int = 2,
    runtime_margin: float = 1.25,  # 在权重/KV 之外预留 25% 运行时空间；按目标后端峰值实测，过低会增加 OOM 风险。
) -> dict[str, float]:
    """估算给定模型与请求形状的权重、KV Cache 和运行余量显存，返回各项 GiB 预算。"""
    # 权重按量化位宽估算；KV Cache 同时包含每层的 Key 和 Value。
    weight_bytes = parameters * weight_bits / 8
    kv_bytes = layers * 2 * batch_size * sequence_length * kv_heads * head_dim * kv_bytes_per_element
    subtotal = weight_bytes + kv_bytes
    return {
        "weights_GiB": my_gib(weight_bytes),
        "kv_cache_GiB": my_gib(kv_bytes),
        "subtotal_GiB": my_gib(subtotal),
        "budget_with_margin_GiB": my_gib(subtotal * runtime_margin),
    }


# 7B、4-bit、32 层、Batch 8、4096 Token、8 KV 头×128 维构成 GQA 容量夹具；生产全部替换为实际 Config 与长度分布。
example_memory_contract = {
    "parameters": 7_000_000_000,
    "weight_bits": 4,
    "layers": 32,
    "batch_size": 8,
    "sequence_length": 4096,
    "kv_heads": 8,
    "head_dim": 128,
}
example_budget = my_estimate_inference_memory(**example_memory_contract)
print({name: round(value, 3) for name, value in example_budget.items()})


#### 3.3.1．KV Cache 的三个增长轴

**学习问题**：模型结构与数据类型固定后，并发 Batch、Prompt 上下文和已生成 Token 分别如何改变 KV Cache 容量？

**验收不变量**：`my_estimate_inference_memory` 中的 KV Cache 必须与 `B×S` 成正比，其中 `S = context_tokens + generated_tokens`。Batch 扩大为原来的 `k` 倍时，KV Cache 也应扩大为 `k` 倍；固定其他变量时，上下文或生成长度增加都应使容量单调线性增长。


In [ ]:
# 直接复用本章容量估算器，在固定模型契约下分离 Batch、上下文和生成长度三个增长轴。
import matplotlib.pyplot as plt

kv_model_contract = {
    name: value
    for name, value in example_memory_contract.items()
    if name not in {"batch_size", "sequence_length"}
}


def my_kv_cache_gib(batch_size: int, context_tokens: int, generated_tokens: int) -> float:
    """在固定模型结构下估算指定 Batch、上下文与生成长度对应的 KV Cache GiB。"""
    estimate = my_estimate_inference_memory(
        **kv_model_contract,
        batch_size=batch_size,
        sequence_length=context_tokens + generated_tokens,
    )
    return estimate["kv_cache_GiB"]


kv_batch_sizes = [1, 2, 4, 8, 16]
kv_context_lengths = [256, 512, 1024, 2048, 4096]
kv_generated_lengths = [0, 128, 256, 512, 1024, 2048]
kv_by_batch = [my_kv_cache_gib(batch, 2048, 512) for batch in kv_batch_sizes]
kv_by_context = [my_kv_cache_gib(8, context, 512) for context in kv_context_lengths]
kv_by_generated = [my_kv_cache_gib(8, 1024, generated) for generated in kv_generated_lengths]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
growth_views = [
    (axes[0], kv_batch_sizes, kv_by_batch, "Batch Size", "上下文 2,048 + 生成 512"),
    (axes[1], kv_context_lengths, kv_by_context, "Prompt 上下文 Token", "Batch 8 + 生成 512"),
    (axes[2], kv_generated_lengths, kv_by_generated, "已生成 Token", "Batch 8 + 上下文 1,024"),
]
for axis, x_values, y_values, x_label, subtitle in growth_views:
    axis.plot(x_values, y_values, marker="o", linewidth=2, color="#0072B2")
    axis.fill_between(x_values, y_values, alpha=0.16, color="#0072B2")
    axis.set_xlabel(x_label)
    axis.set_ylabel("KV Cache（GiB）")
    axis.set_title(subtitle)
    axis.grid(alpha=0.25)
fig.suptitle("固定模型结构下 KV Cache 随并发与已缓存序列长度线性增长", fontsize=14)
fig.tight_layout()
plt.show()

kv_growth_checks = {
    "batch_16x_capacity_16x": math.isclose(kv_by_batch[-1], 16 * kv_by_batch[0]),
    "context_linear_in_total_length": all(
        math.isclose(value / (context + 512), kv_by_context[0] / (kv_context_lengths[0] + 512))
        for context, value in zip(kv_context_lengths, kv_by_context)
    ),
    "generated_linear_in_total_length": all(
        math.isclose(value / (1024 + generated), kv_by_generated[0] / 1024)
        for generated, value in zip(kv_generated_lengths, kv_by_generated)
    ),
    "cached_length_contract": "sequence_length = context_tokens + generated_tokens",
}
print(kv_growth_checks)


**应观察结论**：三幅图在其他条件固定时均呈直线增长。左图说明并发序列共同占用 KV；中图说明 Prefill 完成后长 Prompt 已形成较大的 Cache；右图说明 Decode 每生成一个 Token 仍会继续追加各层的 K/V。

**不可误读边界**：该图是基于本章公式和固定模型契约得到的容量证据，不是设备峰值显存或延迟实测。运行时还包含权重、激活、工作区、碎片和调度预留；Paged KV 改善分配与回收，但不会消除有效 K/V 本身随 `B×S` 增长的事实。


### 3.4．批处理、Padding 与长度分桶

普通批处理把多个请求组成 `[B, T]`：

- 同一批次的 Prefill 可并行，提高硬件利用率。
- 长短请求混合会产生 Padding 浪费；按提示长度分桶（bucketing）可减少浪费。
- Decode 阶段不同请求结束时间不同，静态批次会留下空槽。

批处理验证以 **有效输出 Token/s** 为核心，同时报告批大小、输入/输出长度分布和显存峰值。


In [ ]:
# Left Padding（左填充）示例；生成模型常用左填充对齐最后一个有效 Token。
def my_left_pad_sequences(sequences: list[list[int]], pad_token_id: int):
    """把变长 Token 序列左填充到批内最大长度，返回 input_ids 与 attention_mask 张量。"""
    max_len = max(map(len, sequences))
    input_ids, attention_masks = [], []
    # 在左侧补 PAD，使每个序列最后一个位置都对应真实 token。
    for sequence in sequences:
        pad_len = max_len - len(sequence)
        input_ids.append([pad_token_id] * pad_len + sequence)
        attention_masks.append([0] * pad_len + [1] * len(sequence))
    return torch.tensor(input_ids), torch.tensor(attention_masks)


prompts = [
    [tokenizer.bos_token_id] + tokenizer.encode(my_format_prompt("什么是吞吐量？")),
    [tokenizer.bos_token_id] + tokenizer.encode(my_format_prompt("KV Cache 有什么作用？")),
]
batched_ids, batched_mask = my_left_pad_sequences(prompts, tokenizer.pad_token_id)
print("shape:", tuple(batched_ids.shape), "padding tokens:", int((batched_mask == 0).sum()))


### 3.5．Continuous Batching 的状态转换

**Continuous Batching（连续批处理）**无需等待整个静态批次完成，而是在每个调度步移除已完成请求并接纳新请求，从而提高高并发服务的 Token 级利用率。

本节以确定性状态机呈现调度语义，每行对应一个调度 Tick；该实现不承担完整推理服务器职责。真实引擎还要管理 KV Block、抢占、前缀缓存、Chunked Prefill（分块预填充）、优先级、显存水位和公平性。

<!-- diagram:continuous-batching -->
Continuous Batching 在每个解码步重新编排活跃请求，让新请求可以加入、已完成请求立即释放资源：

```mermaid
flowchart LR
    Q["等待队列"] --> S["调度器"]
    S --> P["新请求 Prefill"]
    S --> D["活跃请求 Decode"]
    P --> B["动态 Batch"]
    D --> B
    B --> G["GPU Forward"]
    G --> T["每个请求生成 1 token"]
    T --> C{"是否结束？"}
    C -->|"否"| D
    C -->|"是"| R["释放 KV blocks"]
    R --> S
```


In [ ]:
# 连续批处理调度模拟器。
@dataclass
class MyRequest:
    """描述连续批处理模拟中的请求到达时间、目标输出长度和已生成进度。"""
    request_id: str
    arrival_tick: int
    output_tokens: int
    generated: int = 0


# capacity=2 仅呈现动态进入/退出；生产按显存压测、长度分布与 SLO 选择并发及 Batch Token 上限。
def my_simulate_continuous_batching(requests: list[MyRequest], capacity: int = 2):
    """按 Tick 接纳请求、推进活跃序列并释放完成槽位，返回确定性的调度时间线。"""
    waiting = sorted(copy.deepcopy(requests), key=lambda item: item.arrival_tick)
    active: list[MyRequest] = []
    timeline = []
    tick = 0

    # 重复执行当前步骤，直到达到停止条件或生成完成。
    while waiting or active:
        while waiting and waiting[0].arrival_tick <= tick and len(active) < capacity:
            active.append(waiting.pop(0))

        stepped = [request.request_id for request in active]
        for request in active:
            request.generated += 1
        finished = [request.request_id for request in active if request.generated >= request.output_tokens]
        active = [request for request in active if request.generated < request.output_tokens]
        timeline.append({"tick": tick, "stepped": stepped, "finished": finished})
        tick += 1

    return timeline


# (到达 Tick, 输出 Token)=(0,2)/(0,5)/(1,2) 是固定调度夹具，不代表服务流量。
requests = [
    MyRequest("A", arrival_tick=0, output_tokens=2),
    MyRequest("B", arrival_tick=0, output_tokens=5),
    MyRequest("C", arrival_tick=1, output_tokens=2),
]
# 保存确定性状态记录，供打印验证与后续时间线共用。
continuous_timeline = my_simulate_continuous_batching(requests, capacity=2)
for row in continuous_timeline:
    print(row)


#### 3.5.1．请求进入、等待、执行与释放时间线

**学习问题**：当动态 Batch 容量为 `2` 时，已完成请求释放的槽位何时能够被等待请求复用？

**验收不变量**：每个 Tick 的 `stepped` 请求数不得超过容量；请求只能在到达后执行；请求生成的步数必须等于其 `output_tokens`；A 在 Tick 1 完成后，等待中的 C 应在下一个 Tick 进入执行，而不必等待 B 完成。


In [ ]:
# 直接消费连续批处理状态机产生的记录，构造请求级等待与执行时间线。
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

timeline_ticks = [row["tick"] for row in continuous_timeline]
timeline_request_ids = [request.request_id for request in requests]
timeline_step_ticks = {
    request_id: [row["tick"] for row in continuous_timeline if request_id in row["stepped"]]
    for request_id in timeline_request_ids
}
timeline_finish_ticks = {
    request_id: next(
        row["tick"] for row in continuous_timeline if request_id in row["finished"]
    )
    for request_id in timeline_request_ids
}
timeline_status = []
for request in requests:
    first_step = timeline_step_ticks[request.request_id][0]
    request_status = []
    for tick in timeline_ticks:
        if request.arrival_tick <= tick < first_step:
            request_status.append(1)  # 已到达但因容量不足而等待。
        elif tick in timeline_step_ticks[request.request_id]:
            request_status.append(2)  # 本 Tick 进入动态 Batch 并生成一个 Token。
        else:
            request_status.append(0)
    timeline_status.append(request_status)

fig, axis = plt.subplots(figsize=(10, 4.2))
timeline_cmap = ListedColormap(["#F4F4F4", "#E69F00", "#0072B2"])
axis.imshow(timeline_status, cmap=timeline_cmap, vmin=0, vmax=2, aspect="auto", interpolation="nearest")
axis.set_xticks(range(len(timeline_ticks)))
axis.set_xticklabels(timeline_ticks)
axis.set_yticks(range(len(timeline_request_ids)))
axis.set_yticklabels(timeline_request_ids)
axis.set_xlabel("调度 Tick")
axis.set_ylabel("请求")
axis.set_title("Continuous Batching：完成请求释放槽位后，等待请求在下一 Tick 加入")
for row_index, request in enumerate(requests):
    for generated_index, tick in enumerate(timeline_step_ticks[request.request_id], start=1):
        axis.text(tick, row_index, str(generated_index), ha="center", va="center", color="white")
    axis.scatter(request.arrival_tick, row_index, marker=">", s=90, facecolors="none", edgecolors="black")
    axis.scatter(timeline_finish_ticks[request.request_id], row_index, marker="x", s=90, color="black")
axis.set_xticks([value - 0.5 for value in range(1, len(timeline_ticks))], minor=True)
axis.set_yticks([value - 0.5 for value in range(1, len(timeline_request_ids))], minor=True)
axis.grid(which="minor", color="white", linewidth=1.5)
axis.tick_params(which="minor", bottom=False, left=False)
axis.legend(
    handles=[
        Patch(facecolor="#E69F00", label="等待槽位"),
        Patch(facecolor="#0072B2", label="本 Tick 生成 1 Token"),
        Line2D([0], [0], marker=">", color="black", markerfacecolor="none", linestyle="None", label="到达"),
        Line2D([0], [0], marker="x", color="black", linestyle="None", label="完成并释放"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.2),
    ncol=4,
)
fig.tight_layout()
plt.show()

timeline_checks = {
    "capacity_respected": max(len(row["stepped"]) for row in continuous_timeline) <= 2,
    "arrival_respected": all(
        all(tick >= request.arrival_tick for tick in timeline_step_ticks[request.request_id])
        for request in requests
    ),
    "generated_steps_match_budget": all(
        len(timeline_step_ticks[request.request_id]) == request.output_tokens for request in requests
    ),
    "C_reuses_A_slot_next_tick": timeline_step_ticks["C"][0] == timeline_finish_ticks["A"] + 1,
}
print(timeline_checks)


**应观察结论**：A 与 B 在 Tick 0 同时进入容量为 2 的动态 Batch；C 在 Tick 1 已到达但需要等待。A 在 Tick 1 完成并释放槽位后，C 在 Tick 2 加入，随后 C 先于长请求 B 完成，空槽位立即回收。格内数字表示该请求生成的第几个 Token。

**不可误读边界**：Tick 是状态机的离散调度步，不代表等长的真实时间；该图证明进入、退出与容量复用语义，不证明 Continuous Batching 一定改善 TTFT、TPOT、吞吐或尾延迟。生产结论仍需消费真实到达分布并执行端到端基准，同时评估公平性、抢占和 KV Block 水位。


## 4．证据验证

原理实现通过四类证据建立语义基线：固定贪心解码下的 Token 序列一致性、KV Cache 容量公式与张量形状、批处理中的有效 Token 吞吐，以及 Continuous Batching 的请求进入、退出与资源释放状态。后续库迁移均以这些结果为比较基准。

| 验证对象 | 固定条件 | 通过判据 |
|---|---|---|
| 朴素生成与 KV Cache | 模型权重、提示、贪心解码、最大输出长度 | 逐 Token 输出一致 |
| KV 容量模型 | 层数、并发、缓存长度、KV Head、dtype | 公式结果与实际张量字节数一致 |
| 批处理 | 相同输入/输出长度分布与设备 | 同时报告有效 Token/s、峰值内存与尾延迟 |
| 连续批处理 | 固定到达时间、输出长度与容量 | 请求按预算进入，完成后立即释放槽位 |


## 5．迁移到生产库

### 5.1．PyTorch SDPA

**SDPA（Scaled Dot-Product Attention，缩放点积注意力）**统一了注意力接口，并会根据设备、dtype、形状和可用后端选择合适实现。

SDPA 接收形状通常为 `[B, H, T, D]` 的 `q/k/v`，输出与原理公式 $\operatorname{softmax}\left(QK^{\top}/\sqrt{D}\right)V$ 相同形状的结果。后端可以避免显式物化大型分数矩阵，并降低自定义 Mask 的错误风险。关闭 Dropout 后，输出一致性验证应以小型 FP32 张量为起点，随后在目标设备上测量性能。


In [ ]:
# 验证原理 Attention 与 SDPA 的数值对齐。
torch.manual_seed(SEED)
# 2×4×8×16 依次为 Batch、Head、序列和 Head Dim；只验证接口，不代表生产形状。
# 固定形状：q.shape = [2, 4, 8, 16]。
q = torch.randn(2, 4, 8, 16)
# 固定形状：k.shape = [2, 4, 8, 16]。
k = torch.randn(2, 4, 8, 16)
# 固定形状：v.shape = [2, 4, 8, 16]。
v = torch.randn(2, 4, 8, 16)

scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
causal = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1)
manual = torch.softmax(scores.masked_fill(causal, float("-inf")), dim=-1) @ v
sdpa = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)

# 该门禁适用于 FP32；低精度或其他 Backend 应依据参考误差分布重新设定。
torch.testing.assert_close(manual, sdpa, rtol=1e-5, atol=1e-6)
print("max_abs_error =", float((manual - sdpa).abs().max()))


### 5.2．FlashAttention 与 HBM 访问

**FlashAttention**通过分块（tiling）与在线 Softmax，减少注意力中间矩阵对高带宽显存（HBM）的读写；数学结果仍是精确注意力（允许正常浮点误差）。

适用性检查包括：

1. GPU 架构、CUDA、PyTorch 与 flash-attn 版本兼容；
2. 模型实现支持对应 attention backend；
3. dtype、head dimension、mask 形式与 dropout 满足后端条件；
4. 无法命中 fused kernel 时是否静默回退；
5. 用目标长度分布同时测 TTFT、TPOT、峰值显存和正确性。

在现代 Transformers 中，优先通过 `attn_implementation="sdpa"` 或受支持的 FlashAttention 后端切换，不应直接重写模型内部注意力。


In [ ]:
# Transformers Attention backend。默认不下载模型。
RUN_FLASH_ATTENTION = False

if RUN_FLASH_ATTENTION:
    from transformers import AutoModelForCausalLM

    # 实例化当前阶段的模型结构，并准备进入训练或推理模式。
    flash_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B",
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",
    )
else:
    print("RUN_FLASH_ATTENTION=False；先以 SDPA 建立兼容基线，再在目标 GPU 上验证 FlashAttention。")


### 5.3．量化模型的推理执行

本节不重新推导 Scale、Zero Point、Calibration 或量化误差；这些属于 [A60_model_compression.ipynb](A60_model_compression.ipynb)。推理侧关注量化资产能否被目标 Kernel 直接消费。

| 执行格式 | 典型用途 | 推理侧确认项 |
|---|---|---|
| BF16/FP16 | GPU 通用基线 | Tensor Core、Attention Backend、权重布局 |
| INT8 Weight-only | 较稳健的权重压缩 | 反量化是否融合进 GEMM、Batch Size 是否合适 |
| INT4 Weight-only | 更低权重带宽 | Group Size、Scale 布局、目标硬件 Kernel |
| FP8 Weight/Activation | 新一代 GPU 推理 | Calibration、累加精度、硬件原生支持 |
| KV Cache Quantization | 长上下文或高并发 | Cache dtype、Attention Kernel、质量与 TPOT |

更低 bit 不自动带来更低延迟。若运行时先完整反量化再调用普通 Dense GEMM，可能只减小文件或权重显存，并没有获得执行加速。验收需要同时记录质量、峰值显存、TTFT、TPOT 和吞吐。


In [ ]:
# INT8 Dynamic Quantization 用于 CPU Linear 机制验证；位宽、粒度与 Kernel 须在目标硬件上联合验收质量和延迟。
# PyTorch Dynamic Quantization（动态量化）作为 INT8 机制入口。
# 它主要面向 CPU nn.Linear；不同 PyTorch 版本可能将 API 放入 torch.ao.quantization。
cpu_model = copy.deepcopy(baseline_model).cpu().eval()
# 将可选依赖或平台能力隔离处理，不影响其余验证路径。
try:
    quantized_cpu_model = torch.ao.quantization.quantize_dynamic(
        cpu_model,
        {nn.Linear},
        dtype=torch.qint8,
    )
    cpu_prompt = prompt_tensor.cpu()
    fp32_output = my_generate_naive(cpu_model, cpu_prompt, max_new_tokens=4)
    int8_output = my_generate_naive(quantized_cpu_model, cpu_prompt, max_new_tokens=4)
    print({
        "fp32_tokens": fp32_output[0].tolist(),
        "int8_tokens": int8_output[0].tolist(),
        "exact_match": torch.equal(fp32_output, int8_output),
    })
except Exception as exc:
    print("当前后端不支持此动态量化路径；在目标 CPU/PyTorch 上选择受支持后端。", exc)


In [ ]:
# Transformers bitsandbytes 4-bit 推理。默认不下载。
RUN_4BIT_INFERENCE = False

if RUN_4BIT_INFERENCE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    # NF4 配合二次量化压缩权重，BF16 保留计算动态范围；模型、Kernel 或硬件变化后须重验质量、显存与延迟。
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    # 初始化 Tokenizer，并固定文本与 token ID 之间的转换协议。
    int4_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
    int4_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B",
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    print("RUN_4BIT_INFERENCE=False；未联网。")


### 5.4．`torch.compile` 的成本与收益边界

`torch.compile` 可以减少 Python 开销、融合算子并针对形状生成 Kernel，但第一次调用包含编译成本。

**测量边界**：

1. 把冷启动/首次编译与稳态延迟分开；
2. 先验证输出一致；
3. 观察 graph break（图断裂）和重复编译；
4. 对 Decode 这类每步形状变化路径，优先使用框架提供的静态 Cache/编译集成；
5. 小模型、CPU、短任务可能得不偿失。


In [ ]:
# 编译固定形状的单步 forward；失败时保留 eager 基线。
RUN_COMPILE = False

if RUN_COMPILE and hasattr(torch, "compile"):
    # 实例化当前阶段的模型结构，并准备进入训练或推理模式。
    compile_model = copy.deepcopy(baseline_model).eval()
    compiled_forward = torch.compile(compile_model, dynamic=False)
    fixed_ids = probe_batch["input_ids"]
    fixed_mask = probe_batch["attention_mask"]

    eager_logits = compile_model(fixed_ids, attention_mask=fixed_mask).logits
    compile_progress = tqdm(
        total=1, desc="torch.compile 首次编译（底层无细粒度进度）",
        unit="graph", dynamic_ncols=True,
    )
    try:
        compiled_logits = compiled_forward(fixed_ids, attention_mask=fixed_mask).logits
    except Exception:
        compile_progress.close()
        raise
    else:
        compile_progress.update(1)
        compile_progress.close()

    # 冷启动保留 1 次编译成本；稳态预热 2 次后重复 10 次，形状或图变化后必须重测缓存命中。
    cold_ms = my_median_runtime_ms(
        lambda: compiled_forward(fixed_ids, attention_mask=fixed_mask),
        repeats=1,
        warmups=0,
    )
    steady_ms = my_median_runtime_ms(
        lambda: compiled_forward(fixed_ids, attention_mask=fixed_mask),
        repeats=10,
        warmups=2,
    )
    print({"compile_cold_ms": cold_ms, "compile_steady_ms": steady_ms})
else:
    print("RUN_COMPILE=False；避免在不合适环境支付编译成本。")


### 5.5．静态形状与动态形状

- **Static Shape（静态形状）**：固定 Batch、序列长度与 Cache 上限，编译器更容易生成高度特化的 Kernel，但可能增加 Padding 和内存预留。
- **Dynamic Shape（动态形状）**：同一图接受多个长度，减少重复编译与 Padding，但某些优化更难，且仍可能因数据依赖控制流而重新编译。

生产折中常用“有限形状桶”：例如把 Prefill 长度映射到 128/256/512/1024 桶，Decode 使用预分配 Static Cache。任意长度不宜分别编译为独立静态图。


In [ ]:
# 形状分桶与 Padding 浪费估算。
# 64…4096 的 2 次幂桶便于复用编译缓存；桶更密减少 Padding、增加编译与缓存开销。
def my_choose_bucket(length: int, buckets=(64, 128, 256, 512, 1024, 2048, 4096)) -> int:
    """选择首个能够容纳请求长度的形状桶；超过最大桶时抛出 ValueError。"""
    for bucket in buckets:
        if length <= bucket:
            return bucket
    raise ValueError("长度超过最大 bucket")


# 请求长度覆盖桶内与跨边界情况，仅用于验证向上取整。
request_lengths = [37, 91, 120, 257, 900]
bucketed = [(length, my_choose_bucket(length)) for length in request_lengths]
padding = sum(bucket - length for length, bucket in bucketed)
print({"mapping": bucketed, "padding_tokens": padding})


### 5.6．Speculative Decoding

KV Cache 减少每一步的重复计算，但自回归生成仍然一次确认一个 token。Speculative Decoding 使用较小的 Draft Model 一次提出多个候选，再由 Target Model 并行验证：

```mermaid
flowchart LR
    P["当前已接受前缀"] --> D["Draft Model<br/>提出 k 个 token"]
    D --> T["Target Model<br/>一次并行验证"]
    T --> A{"接受多少个连续候选？"}
    A -->|"全部或部分接受"| U["更新输出与 KV Cache"]
    A -->|"出现拒绝"| C["按正确接受规则修正 token"]
    C --> U
    U --> P
```

正确实现的接受/修正规则应保持 Target Model 的目标分布；它不是直接相信小模型。收益主要取决于接受率、Draft 成本、Target 验证效率、序列长度和并发度：

- 低并发、Target 很大且 Draft 命中率高时，TPOT 通常更可能受益。
- 高并发服务已经能用大 Batch 填满 GPU 时，额外 Draft 与验证可能挤占吞吐。
- Draft 和 Target 的 Tokenizer、采样参数、KV 管理与停止条件必须一致。
- 应分别测接受长度、Target Forward 次数、TTFT、TPOT、吞吐和额外显存。


### 5.7．Transformers `generate()`

本地随机初始化的小型 Llama 用于验证 API，实验不依赖外部权重，输出也不用于评价语言质量。生产模型中，`generate()` 统一处理停止条件、Sampling（采样）、Beam Search（束搜索）、Cache 与批处理。

关键约束：

- 用 `max_new_tokens` 控制输出长度，不把提示长度混入生成预算。
- 显式配置 `pad/eos`；批处理时核对 padding side。
- 生产调用优先使用配置对象和版本化默认值，避免库升级改变采样行为。
- 静态 Cache 可能触发或配合编译优化；动态 Cache 更灵活。两者应在真实长度分布上比较。


In [ ]:
# 本地随机模型调用 generate()，不下载权重。
try:
    # 实例化当前阶段的模型结构，并准备进入训练或推理模式。
    local_generate_model = LlamaForCausalLM(hf_config).eval()
    local_prompt = torch.tensor([[tokenizer.bos_token_id] + tokenizer.encode("用户：你好\n助手：")])
    local_generated = local_generate_model.generate(
        input_ids=local_prompt,
        attention_mask=torch.ones_like(local_prompt),
        max_new_tokens=8,  # 8 Token 仅用于确定性回归；长度增加会线性抬高 Decode 与 KV 成本。
        do_sample=False,  # 关闭采样以比较优化路径的 Token 一致性；采样任务须记录完整解码配置。
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    print(local_generated.tolist())
except NameError:
    print("跳过：请先运行 Transformers + PEFT 章节中的本地 hf_config 单元。")


In [ ]:
# 有语义的小模型生成；默认不下载。
RUN_PRETRAINED_GENERATE = False

if RUN_PRETRAINED_GENERATE:
    from transformers import (
        AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList,
    )

    # 整理当前阶段使用的数据子集，避免不同用途的数据混用。
    pretrained_id = "Qwen/Qwen3-0.6B"
    pretrained_tokenizer = AutoTokenizer.from_pretrained(pretrained_id)
    # 实例化当前阶段的模型结构，并准备进入训练或推理模式。
    pretrained_model = AutoModelForCausalLM.from_pretrained(
        pretrained_id,
        dtype="auto",
        device_map="auto",
        attn_implementation="sdpa",
    )
    inputs = pretrained_tokenizer("请用一句话解释 KV Cache。", return_tensors="pt").to(pretrained_model.device)
    class MyPretrainedGenerationProgress(LogitsProcessor):
        """按预训练模型的解码步更新进度，并保持 logits 不变。"""
        def __init__(self, total):
            self.progress = tqdm(
                total=total, desc="生成 KV Cache 说明", unit="token-step", dynamic_ncols=True
            )

        def __call__(self, input_ids, scores):
            self.progress.update(1)
            return scores

        def close(self):
            self.progress.close()

    generation_progress = MyPretrainedGenerationProgress(64)
    try:
        outputs = pretrained_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            cache_implementation="dynamic",
            logits_processor=LogitsProcessorList([generation_progress]),
        )
    finally:
        generation_progress.close()
    print(pretrained_tokenizer.decode(outputs[0], skip_special_tokens=True))
else:
    print("RUN_PRETRAINED_GENERATE=False；未联网。")


### 5.8．TTFT、TPOT 与吞吐基准

基准测试需要定义边界：

- TTFT：包含 Tokenization、排队、Prefill、采样和网络传输中的哪些部分？
- TPOT：排除第一个 Token 后，对稳态 Decode 取平均还是 P50/P95/P99？
- Throughput：按输出 Token、总 Token，还是完成请求数？

本节测量**模型内核路径**，不含网络和 Tokenization。生产服务还应记录排队时间、端到端延迟、错误率、并发度、输入/输出长度分位数与硬件利用率。


In [ ]:
# 对原理模型测量模型内 TTFT、TPOT 与吞吐。
@torch.inference_mode()
def my_benchmark_cached_generation(model: MyTinyCausalLM, prompt_ids: torch.Tensor, new_tokens: int = 16):
    """测量缓存生成的模型内 TTFT、平均 TPOT 与输出吞吐，并返回请求形状元数据。"""
    model.eval()
    my_synchronize(DEVICE)
    total_start = time.perf_counter()

    mask = torch.ones_like(prompt_ids)
    prefill_start = time.perf_counter()
    output = model(prompt_ids, attention_mask=mask, use_cache=True)
    next_token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
    my_synchronize(DEVICE)
    ttft = time.perf_counter() - prefill_start

    generated = [next_token]
    past = output.past_key_values
    decode_times = []
    for _ in range(max(new_tokens - 1, 0)):
        mask = torch.ones(prompt_ids.size(0), prompt_ids.size(1) + len(generated), device=DEVICE, dtype=torch.long)
        start = time.perf_counter()
        output = model(next_token, attention_mask=mask, past_key_values=past, use_cache=True)
        next_token = output.logits[:, -1].argmax(dim=-1, keepdim=True)
        past = output.past_key_values
        my_synchronize(DEVICE)
        decode_times.append(time.perf_counter() - start)
        generated.append(next_token)

    elapsed = time.perf_counter() - total_start
    output_token_count = prompt_ids.size(0) * len(generated)
    return {
        "ttft_ms": ttft * 1000,
        "tpot_ms": statistics.mean(decode_times) * 1000 if decode_times else float("nan"),
        "output_tokens_per_second": output_token_count / elapsed,
        "batch_size": prompt_ids.size(0),
        "prompt_length": prompt_ids.size(1),
        "new_tokens": len(generated),
    }


# 一次预热后重复 5 次取中位数是低成本基线；生产按编译稳定点增加样本并报告 P50/P95/P99 与置信区间。
_ = my_benchmark_cached_generation(baseline_model, prompt_tensor, new_tokens=4)
benchmark_samples = [
    my_benchmark_cached_generation(baseline_model, prompt_tensor, new_tokens=16)
    for _ in trange(5, desc="采集推理指标", unit="run", dynamic_ncols=True)
]
timing_keys = ("ttft_ms", "tpot_ms", "output_tokens_per_second")
benchmark_medians = {
    key: statistics.median(sample[key] for sample in benchmark_samples)
    for key in timing_keys
}
benchmark_medians.update({
    key: benchmark_samples[0][key]
    for key in ("batch_size", "prompt_length", "new_tokens")
})
print(benchmark_medians)


### 5.9．Replica、TP、PP 与混合部署

训练并行与推理并行复用部分技术，但优化目标不同：推理没有反向传播，却要满足 TTFT/TPOT、并发与可用性。

| 部署方式 | 模型副本 | 单请求跨卡 | 主要收益 | 主要代价 |
|---|---:|---:|---|---|
| Replica / Data Parallel | 每组完整副本 | 否 | 水平扩吞吐、故障隔离简单 | 单副本必须放得下模型 |
| Tensor Parallel | 一组内分片 | 是 | 降低单卡权重与单步计算 | 每层通信，强依赖高速互联 |
| Pipeline Parallel | 按层分 Stage | 是 | 跨设备容纳深模型 | 单请求流水线气泡、调度复杂 |
| Expert Parallel | MoE Expert 分布 | 是 | 容纳大量专家 | All-to-All 与负载不均 |
| Replica × TP | 多个 TP 组 | 是 | 同时解决容量和吞吐 | 路由与容量规划更复杂 |

**选择原则**：模型在单卡满足 SLO 时优先使用 Replica；容量不足或单卡 TPOT 不达标时评估节点内 TP；只有层级放置或跨节点容量确有需要时再引入 PP。扩大单请求并行度会增加 Collective 延迟，因此并行规模应由容量和性能证据决定。

#### 5.9.1．Head 数与 TP 并行度的理论边界

以下讨论限定为常规 MHA、GQA 与 MQA 的 Head Parallel 实现。设 Query Head 数为 $H_q$、KV Head 数为 $H_{kv}$、隐藏维度为 $D$、TP 并行度为 $P$。模型结构首先要满足：

$$D_h = D / H_q \in \mathbb{N}, \qquad H_q / H_{kv} \in \mathbb{N}$$

标准 Head Parallel 把完整 Head 均匀分配给各 Rank，因此每个 Rank 的 Query Head 数为 $H_q/P$，要求 $H_q \bmod P=0$。若 KV Head 也做等量分片，则还要求 $H_{kv} \bmod P=0$。这些是特定分片布局的整数形状约束，不是张量并行的数学定义。

理论上，不能整除仍可通过不均匀分片、补齐后丢弃 Padding、沿 Head Dimension 继续切分或增加重排与 Collective 保持精确语义；代价是负载不均、无效计算、额外通信以及 Kernel 形状复杂化。因此，‘理论上可以 TP’不能推出‘目标运行时能够加载’，也不能推出‘性能值得采用’。

#### 5.9.2．MHA、GQA 与 MQA 的运行时差异

| Attention 结构与策略 | Query Head | KV Head | 每卡 KV 比例 | 生产含义 |
|---|---|---|---:|---|
| MHA 等量分片 | $H_q \bmod P=0$ | $H_{kv}=H_q$，随 Query 均分 | $1/P$ | 标准路径，形状与负载最规则 |
| GQA 等量分片 | $H_q \bmod P=0$ | $H_{kv} \bmod P=0$ | $1/P$ | KV Cache 随 TP 下降 |
| GQA/MQA KV 复制 | $H_q \bmod P=0$ | 当 $P > H_{kv}$ 且 $P \bmod H_{kv}=0$ 时，每个 KV Head 可复制到多个 Rank | $1/H_{kv}$ | 可以执行，但 KV Cache 不再按 $1/P$ 下降 |
| 任意不整除分片 | 依赖自定义布局 | 依赖自定义布局 | 依赖实现 | 标准引擎通常拒绝；需要专用 Kernel、Padding 或改用 PP |

KV 复制的复制因子为 $R_{kv}=P/H_{kv}$。例如 $H_q=32$、$H_{kv}=8$、$P=16$ 时，每卡仍有 2 个 Query Head，但一个 KV Head 会由两个 Rank 复制保存；每卡 KV Cache 是完整副本的 $1/8$，而不是 $1/16$。因此容量模型若一律按 `KV / TP` 计算，会低估显存。

```mermaid
flowchart TD
    A[读取模型 Config 与目标运行时版本] --> B{隐藏维度与 GQA 分组是否合法}
    B -->|否| X[拒绝：模型结构契约不成立]
    B -->|是| C{Query Head 能被 TP 整除}
    C -->|否| Y[标准 Head Parallel 不可用<br/>评估 Padding、自定义分片或 PP]
    C -->|是| D{KV Head 能被 TP 整除}
    D -->|是| E[Q/K/V 均匀分片]
    D -->|否| F{运行时支持且 TP 能按 KV Head 成组复制}
    F -->|是| G[复制 KV Head<br/>重算显存与吞吐]
    F -->|否| Z[拒绝该 TP 配置或降低 TP]
    E --> H[实测质量、峰值显存、通信占比、TTFT 与 TPOT]
    G --> H
```

运行时准入必须使用锁定版本做预检。Head 兼容只是必要条件之一；Hidden/FFN/Vocabulary 分片、量化 Group Size、MoE Expert 拓扑、Attention Backend、Checkpoint 布局和设备互联仍可能使配置失败或性能退化。下方 GQA 夹具运行后，TP=1/4/8 应显示 KV 均匀分片，TP=16 应显示 KV 复制因子为 2，且每卡 KV 仍为完整 Cache 的 $1/8$；两个失败夹具应分别拒绝不能整除的 Query Head 和禁用复制后的 KV Head 配置。

<!-- diagram:serving-parallel-topology -->
生产拓扑通常先用 Replica 扩吞吐，再在单模型放不下或单卡算力不足时引入 Tensor Parallel：

![架构图：全局 Router 下两个 Replica 与各自 Tensor Parallel GPU 分片组](assets/figures/A50_inference_optimization/serving-parallel-topology.svg)

[TikZ 源文件](assets/figures/A50_inference_optimization/serving-parallel-topology.tex)


In [ ]:
# 校验常规 MHA/GQA/MQA 的标准 Head Parallel 兼容性。
def my_tp_attention_compatibility(
    hidden_size: int,
    num_attention_heads: int,
    num_key_value_heads: int,
    tensor_parallel_size: int,
    kv_policy: str = "replicate_when_needed",
) -> dict[str, int | float | bool | str]:
    """返回标准 Head Parallel 的形状兼容性与每卡 KV 比例，不替代目标运行时预检。"""
    values = (hidden_size, num_attention_heads, num_key_value_heads, tensor_parallel_size)
    if any(value <= 0 for value in values):
        raise ValueError("模型维度、Head 数和 TP 大小必须为正整数")
    if kv_policy not in {"strict_shard", "replicate_when_needed"}:
        raise ValueError("kv_policy 仅支持 strict_shard 或 replicate_when_needed")

    if hidden_size % num_attention_heads != 0:
        return {"supported": False, "reason": "hidden_size 不能被 Query Head 数整除"}
    if num_attention_heads % num_key_value_heads != 0:
        return {"supported": False, "reason": "Query Head 数不能按 KV Head 均匀分组"}
    if num_attention_heads % tensor_parallel_size != 0:
        return {"supported": False, "reason": "Query Head 数不能被 TP 大小整除"}

    if num_key_value_heads % tensor_parallel_size == 0:
        local_kv_heads = num_key_value_heads // tensor_parallel_size
        kv_mode = "sharded"
        kv_replication_factor = 1
    elif (
        kv_policy == "replicate_when_needed"
        and tensor_parallel_size > num_key_value_heads
        and tensor_parallel_size % num_key_value_heads == 0
    ):
        local_kv_heads = 1
        kv_mode = "replicated"
        kv_replication_factor = tensor_parallel_size // num_key_value_heads
    else:
        return {
            "supported": False,
            "reason": "KV Head 既不能均匀分片，也不能按当前策略成组复制",
        }

    return {
        "supported": True,
        "reason": "标准 Head Parallel 形状兼容",
        "head_dim": hidden_size // num_attention_heads,
        "local_query_heads": num_attention_heads // tensor_parallel_size,
        "local_kv_heads": local_kv_heads,
        "kv_mode": kv_mode,
        "kv_replication_factor": kv_replication_factor,
        "per_gpu_kv_fraction": local_kv_heads / num_key_value_heads,
    }


# 粗略比较 Replica × TP 方案的副本数与理论显存。
def my_serving_parallel_plan(
    gpu_count: int,
    tensor_parallel_size: int,
    hidden_size: int,
    num_attention_heads: int,
    num_key_value_heads: int,
    model_weight_gib: float,
    kv_cache_gib_per_replica: float,
    gpu_memory_gib: float,
    runtime_margin_gib: float = 3.0,  # 每卡预留 3 GiB 仅为规划夹具；按框架工作区和碎片峰值校准。
    kv_policy: str = "replicate_when_needed",
) -> dict[str, int | float | bool | str]:
    """估算 Replica × TP 容量；Head 或 GPU 拓扑不兼容时拒绝配置。"""
    if tensor_parallel_size <= 0 or gpu_count <= 0:
        raise ValueError("GPU 数和 TP 大小必须为正整数")
    if gpu_count % tensor_parallel_size != 0:
        raise ValueError("GPU 数必须能被 TP 大小整除")
    if model_weight_gib < 0 or kv_cache_gib_per_replica < 0 or runtime_margin_gib < 0:
        raise ValueError("权重、KV Cache 与运行时余量不得为负数")
    if gpu_memory_gib <= 0:
        raise ValueError("单卡显存必须为正数")

    attention_contract = my_tp_attention_compatibility(
        hidden_size=hidden_size,
        num_attention_heads=num_attention_heads,
        num_key_value_heads=num_key_value_heads,
        tensor_parallel_size=tensor_parallel_size,
        kv_policy=kv_policy,
    )
    if not attention_contract["supported"]:
        raise ValueError(f"TP 与 Attention 不兼容：{attention_contract['reason']}")

    # 权重按 TP 均分；KV Cache 按每卡实际 KV Head 比例计算，复制时不再假设为 1/TP。
    replicas = gpu_count // tensor_parallel_size
    per_gpu_weights = model_weight_gib / tensor_parallel_size
    per_gpu_kv = kv_cache_gib_per_replica * attention_contract["per_gpu_kv_fraction"]
    required = per_gpu_weights + per_gpu_kv + runtime_margin_gib
    return {
        "replicas": replicas,
        "tensor_parallel_size": tensor_parallel_size,
        "local_query_heads": attention_contract["local_query_heads"],
        "local_kv_heads": attention_contract["local_kv_heads"],
        "kv_mode": attention_contract["kv_mode"],
        "kv_replication_factor": attention_contract["kv_replication_factor"],
        "estimated_per_gpu_weight_GiB": per_gpu_weights,
        "estimated_per_gpu_kv_GiB": per_gpu_kv,
        "estimated_per_gpu_GiB": required,
        "fits": required <= gpu_memory_gib,
    }


# 16 卡、32 Query Head、8 KV Head 的 GQA 夹具展示均分与复制边界；生产必须读取实际 Config。
for tp_size in (1, 4, 8, 16):
    print(my_serving_parallel_plan(
        gpu_count=16,
        tensor_parallel_size=tp_size,
        hidden_size=4096,
        num_attention_heads=32,
        num_key_value_heads=8,
        model_weight_gib=28.0,
        kv_cache_gib_per_replica=8.0,
        gpu_memory_gib=24.0,
    ))


print(my_tp_attention_compatibility(
    hidden_size=3840,
    num_attention_heads=30,
    num_key_value_heads=10,
    tensor_parallel_size=4,
))
print(my_tp_attention_compatibility(
    hidden_size=4096,
    num_attention_heads=32,
    num_key_value_heads=8,
    tensor_parallel_size=16,
    kv_policy="strict_shard",
))


### 5.10．权重、适配器与 KV Cache 的分层

**Storage-Compute Disaggregation（存算分离）**不是把所有数据都放到远端存储，而是把“权威持久化、节点缓存、GPU 热数据、计算实例”分层管理。

<!-- diagram:serving-storage-hierarchy -->

![架构图：模型权重、Adapter 与 KV Cache 从权威存储到 GPU 热数据的层级](assets/figures/A50_inference_optimization/serving-storage-hierarchy.svg)

[TikZ 源文件](assets/figures/A50_inference_optimization/serving-storage-hierarchy.tex)

模型版本、Adapter 版本、请求与 Cache Key 共同决定计算节点加载的不可变制品。持久制品与计算实例解耦后，节点可以弹性扩缩容，本地缓存也能降低冷启动期间的远端读取成本。验证覆盖冷/热启动时间、Cache 命中率、远端带宽、哈希失败行为、并发加载、回滚与节点故障恢复。

架构风险：直接让所有 Worker 同时从对象存储加载数十 GB 权重会形成启动风暴。生产方案应使用不可变版本、分层缓存、并发限速、断点续传、内容哈希、预热和就绪探针；未完成加载的实例不得接流量。


#### 5.10.1．KV Cache 分层与外置

KV Cache 是请求级、随序列增长的高带宽状态。可按热度分层：

1. GPU HBM：最低 TPOT，容量最贵；
2. CPU DRAM：容量较大，但换入/换出会增加延迟并占用 PCIe/NVLink 带宽；
3. 本地 NVMe：适合更冷的前缀或可恢复状态；
4. 远端 KV Store：适合跨实例复用、Prefill/Decode 解耦或故障迁移，但网络传输成本可能高于重算。

Cache Key 必须至少绑定模型 ID、权重哈希、Adapter、Tokenizer/模板、RoPE/位置配置、精度与完整 Token 前缀；键不完整会导致静默错误输出。多租户环境还要做隔离、加密、TTL 与敏感提示清除。


In [ ]:
# 判断“传 KV”是否可能比“重算 Prefill”更划算。
def my_kv_transfer_analysis(
    kv_cache_gib: float,
    effective_bandwidth_gib_s: float,
    network_rtt_ms: float,
    recompute_prefill_ms: float,
    serialization_overhead_ms: float = 0.0,
) -> dict[str, float | bool]:
    """比较跨节点传输 KV Cache 与重算 Prefill 的耗时，并给出盈亏平衡有效带宽。"""
    # 传输耗时由数据量/有效带宽、网络 RTT 和序列化开销共同组成。
    transfer_ms = (
        kv_cache_gib / effective_bandwidth_gib_s * 1000
        + network_rtt_ms
        + serialization_overhead_ms
    )
    return {
        "kv_transfer_ms": transfer_ms,
        "recompute_prefill_ms": recompute_prefill_ms,
        "transfer_is_faster": transfer_ms < recompute_prefill_ms,
        "break_even_bandwidth_GiB_s": kv_cache_gib / max((recompute_prefill_ms - network_rtt_ms - serialization_overhead_ms) / 1000, 1e-9),
    }


# 2 GiB KV、20 GiB/s 有效带宽、1 ms RTT、180 ms 重算与 4 ms 序列化均须由真实前缀和网络拓扑实测。
print(my_kv_transfer_analysis(
    kv_cache_gib=2.0,
    effective_bandwidth_gib_s=20.0,
    network_rtt_ms=1.0,
    recompute_prefill_ms=180.0,
    serialization_overhead_ms=4.0,
))


#### 5.10.2．Prefill/Decode 分离（P/D Disaggregation）

Prefill 是计算密集、并行度高的矩阵运算；Decode 是内存带宽敏感、逐 Token 的小步运算。把两者放在不同 Worker 池可以独立扩缩容和隔离长 Prompt 对稳态 Decode 尾延迟的干扰。

```mermaid
sequenceDiagram
    participant R as Router
    participant P as Prefill Pool
    participant K as KV Transport/Store
    participant D as Decode Pool
    R->>P: Prompt + model/adapter version
    P->>P: Prefill
    P->>K: 分层传输 KV blocks
    K->>D: KV handle / blocks
    D->>D: Token-by-token decode
    D-->>R: Streaming tokens
```

**验证指标**：P/D 队列时间、KV 传输字节和耗时、Decode Worker 等待 KV 的比例、TTFT、Inter-Token Latency（Token 间延迟）P99、端到端吞吐和失败恢复。P/D 分离主要用于隔离干扰与控制尾延迟，不应预设它一定提高总吞吐；网络与 KV 传输可能成为新瓶颈。


## 6．生产边界

1. **固定输入资产**：锁定模型、Tokenizer、Adapter、GenerationConfig 和质量集；压缩模型必须带上格式与 Calibration 元数据。
2. **建立执行基线**：使用 Eager + BF16/FP16 + 标准 SDPA，分开记录排队、Prefill、Decode 和网络耗时。
3. **定位主瓶颈**：
   - TTFT 高：检查 Prompt 长度、Prefix Cache、Prefill Kernel、Chunked Prefill 和 P/D 分离。
   - TPOT 高：检查 KV Cache、内存带宽、量化 Kernel、编译、Speculative Decoding 与单请求并行。
   - 高并发吞吐低：检查 Continuous Batching、Paged KV、Batch Token Budget、Replica 和调度空洞。
   - OOM：区分 Weight、KV 和 Workspace，再选择量化执行格式、GQA/MQA、KV 分层、TP/PP 或 Offload。
4. **减少重复工作**：消除重复 Prefill、历史 K/V 重算和 Padding。
5. **优化单次执行**：优化 Attention Backend、低精度 Kernel、融合算子、编译和形状。
6. **提高设备利用率**：引入连续批处理、分页 Cache、Chunked Prefill 和调度策略。
7. **扩展到分布式服务**：单实例达标后再规划 Replica、TP/PP/EP、P/D 分离和 Cache 分层。
8. **交付部署主章**：由部署控制面完成限流、背压、取消、滚动发布、故障隔离与观测。

增加 GPU 数量无法修复单实例低效。若优化需要改变模型本体，应回到模型压缩或训练阶段建立新的版本化产物。

### 6.1．缓存隔离

跨用户 Prefix/Response Cache 必须把租户、权限、模型版本、Adapter、GenerationConfig 和工具状态纳入隔离或 Cache Key；敏感数据不得因提高命中率而越权复用。


### 6.2．端到端实验矩阵

实验每次改变一个主要推理变量，并固定模型语义、提示集合、输出长度和并发分布：

| 实验 | 模型资产 | Cache | 执行与调度 | 质量 | 峰值内存 | TTFT | TPOT | Token/s |
|---|---|---|---|---:|---:|---:|---:|---:|
| A | BF16 基线 | 无 | Eager、静态批次 | 记录 | 记录 | 记录 | 记录 | 记录 |
| B | 同一模型 | Dynamic KV | SDPA、静态批次 | 记录 | 记录 | 记录 | 记录 | 记录 |
| C | 同一模型 | Paged KV | Continuous Batching | 记录 | 记录 | 记录 | 记录 | 记录 |
| D | 量化产物 | Paged KV | 量化 Kernel | 记录 | 记录 | 记录 | 记录 | 记录 |
| E | 同一模型 | Prefix Cache | 专用引擎 + P/D 分离 | 记录 | 记录 | 记录 | 记录 | 记录 |
| F | 同一模型 | Paged KV | 锁定引擎 + 候选 TP | 记录 | 记录 | 记录 | 记录 | 记录 |

量化产物与 BF16 基线首先比较质量，再比较性能；不同模型、提示长度、生成长度或并发度的结果不能直接放在同一列下结论。TP 实验还要固定 Head/KV Head 布局，记录均匀分片或复制模式、每卡 KV 比例、Collective 时间和实际加载结果。


### 6.3．生产验收覆盖

1. 同一批 Prompt 的 Prefill、Decode、排队和端到端耗时。
2. 不同提示长度与输出长度下的朴素生成和 KV Cache 生成的复杂度。
3. Dynamic、Static 与 Paged KV 的显存占用、碎片和编译行为。
4. SDPA、FlashAttention、BF16 与量化 Kernel 的单变量基准矩阵。
5. Continuous Batching 的优先级、取消、最大等待时间与饥饿风险。
6. Prefix Cache 的命中率、节省的 Prefill Token 和租户隔离。
7. Replica 与 TP 方案的 Config 兼容性、每卡 Query/KV Head、KV 分片或复制模式、峰值显存、吞吐、TPOT、通信占比和故障域。
8. P/D 分离的 KV 传输时间、Decode 等待比例和尾延迟。


### 6.4．参考资料

- [PyTorch Scaled Dot-Product Attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- [PyTorch torch.compile](https://docs.pytorch.org/docs/stable/generated/torch.compile.html)
- [PyTorch GPT Fast：KV Cache、编译与 Speculative Decoding](https://pytorch.org/blog/accelerating-generative-ai-2/)
- [PyTorch Speculative Decoding Guide](https://pytorch.org/blog/hitchhikers-guide-speculative-decoding/)
- [Transformers Optimization Overview](https://huggingface.co/docs/transformers/main/optimization_overview)
- [Transformers Attention Interface](https://huggingface.co/docs/transformers/main/attention_interface)
- [vLLM Automatic Prefix Caching](https://docs.vllm.ai/en/v0.10.0/design/automatic_prefix_caching.html)
- [PyTorch Knowledge Distillation Tutorial](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html)
- [PyTorch Tensor Parallel](https://docs.pytorch.org/docs/stable/distributed.tensor.parallel.html)
- [PyTorch Pipeline Parallel](https://docs.pytorch.org/docs/stable/distributed.pipelining.html)
- [vLLM QKVParallelLinear：Query Head 分片与 KV Head 复制](https://docs.vllm.ai/en/latest/api/vllm/model_executor/layers/linear/)
- [vLLM-Omni Tensor Parallel：运行时整除约束示例](https://docs.vllm.ai/projects/vllm-omni/en/latest/design/feature/tensor_parallel/)
- [Megatron Core Parallelism Guide](https://docs.nvidia.com/megatron-core/developer-guide/latest/user-guide/parallelism-guide.html)

生产使用前，以部署环境中锁定版本对应的官方文档为准。


### 6.5．方法总结

推理优化分三层：KV/Prefix/Response Cache 与 Speculative Decoding 减少重复工作或串行步数；低精度 Kernel、SDPA/FlashAttention、融合、编译和形状分桶降低单次执行成本；Continuous Batching、Paged KV、Replica、TP/PP/EP 与 P/D 分离提高设备利用率和容量。

验收需要固定模型语义和真实负载，分别记录排队、Prefill、Decode、TTFT、TPOT、吞吐、峰值显存、Cache 命中、通信和单位有用 Token 成本。服务发布、扩缩容、故障隔离与回滚由 [60_inference_deployment.ipynb](60_inference_deployment.ipynb) 承接。
